In [1]:
import streamlit as st
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import sys

sys.path.append("/Users/toby/Dev/lionel-app/")
from lionel_app.connector import DBManager

from lionel_app.plot_team import create_plot, create_value_plot

In [2]:

dbm = DBManager("/Users/toby/Dev/lionel/data/fpl.db")
# Session = sessionmaker(bind=dbm.engine)

In [3]:
df_sel = pd.DataFrame(dbm.query("SELECT * FROM selections WHERE gameweek = 5").fetchall())


In [4]:
create_plot(df_sel)

In [5]:
create_value_plot(df_sel)

In [10]:
df_team = df_sel.copy()
df_team['name'] = df_team['player'].str.split('_').str[1]
df_team["mean_points_pred"] = df_team["mean_points_pred"].round(1)
df_not_picked = df_team[df_team["picked"] == 0]
df_picked = df_team[df_team["picked"] == 1]
df_picked

,player,team_name,position,value,mean_points_pred,next_points_pred,picked,captain,first_xi,season,picked_time,gameweek,name
0,113_Raya,Arsenal,GK,55,5.2,6.758316,1.0,0,1.0,25,2024-09-17 17:09:01.815409,5,Raya
1,19_Saka,Arsenal,MID,101,6.7,7.309715,1.0,1,1.0,25,2024-09-17 17:09:01.815409,5,Saka
2,29_White,Arsenal,DEF,65,5.4,6.488943,1.0,0,1.0,25,2024-09-17 17:09:01.815409,5,White
3,362_Palmer,Chelsea,MID,106,6.3,6.205915,1.0,0,1.0,25,2024-09-17 17:09:01.815409,5,Palmer
4,506_Pedro Porro,Tottenham,DEF,55,4.2,4.728504,1.0,0,1.0,25,2024-09-17 17:09:01.815409,5,Pedro Porro
5,511_Romero,Tottenham,DEF,51,3.6,4.094542,1.0,0,1.0,25,2024-09-17 17:09:01.815409,5,Romero
6,516_Son,Tottenham,MID,100,6.0,6.381493,1.0,0,1.0,25,2024-09-17 17:09:01.815409,5,Son
7,526_Bowen,West Ham,MID,75,5.0,4.518992,1.0,0,1.0,25,2024-09-17 17:09:01.815409,5,Bowen
8,60_Watkins,Aston Villa,FWD,89,6.0,6.494274,1.0,0,1.0,25,2024-09-17 17:09:01.815409,5,Watkins
9,927_Ndidi,Leicester,MID,50,3.9,3.457511,1.0,0,1.0,25,2024-09-17 17:09:01.815409,5,Ndidi


In [8]:

fig = go.Figure()

# Plot the unpicked players
fig.add_trace(
    go.Scatter(
        x=df_not_picked[df_not_picked["mean_points_pred"] == 0].value,
        y=df_not_picked[df_not_picked["mean_points_pred"] == 0][
            "mean_points_pred"
        ],
        marker=dict(
            color="#9fbbe3",
        ),
        mode="markers",
        customdata=df_not_picked[["name", "team_name", "mean_points_pred"]],
        hovertemplate="<b>%{customdata[0]}</b>"
        + "<br><br><b>Team:</b> %{customdata[1]}"
        + "<br><b>Mean Predicted Points:</b> %{customdata[2]}"
        + "<extra></extra>",
    )
)

# Plot the picked players
fig.add_trace(
    go.Scatter(
        x=df_picked.value,
        y=df_picked["mean_points_pred"],
        marker=dict(
            color="#4B5563",
        ),
        mode="markers",
        customdata=df_picked[["name", "team_name", "mean_points_pred"]],
        hovertemplate="<b>%{customdata[0]}</b>"
        + "<br><br><b>Team:</b> %{customdata[1]}"
        + "<br><b>Mean Predicted Points:</b> %{customdata[2]}"
        + "<extra></extra>",
    )
)

fig.update_layout(
    autosize=False,
    width=700,
    height=800,
    showlegend=False,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    xaxis_title="Player Value (as of previous GW)",
    yaxis_title="Mean Predicted Points",
    yaxis_visible=True,
    yaxis_showticklabels=False,
    xaxis_visible=True,
    xaxis_showticklabels=False,
    font=dict(family="sans-serif", color="#4B5563"),
    margin={"t": 10, "b": 0},
)
return fig

In [8]:
pred_var = 'y_LGBMRegressor_no_exog'
df_plot = df[df[pred_var].notna()]

In [14]:
df_plot

,name,opponent_team_name,is_home,season,gameweek,team_name,position,y,y_Naive,y_LGBMRegressor_no_exog,y_LSTMWithReLU
38,Aaron Ramsdale,Wolves,True,25,1,Arsenal,GK,NaN,0.0,0.149775,0.0
39,Aaron Ramsdale,Aston Villa,False,25,2,Arsenal,GK,NaN,0.0,0.629855,0.0
40,Aaron Ramsdale,Brighton,True,25,3,Arsenal,GK,NaN,0.0,1.117793,0.0
41,Aaron Ramsdale,Tottenham,False,25,4,Arsenal,GK,NaN,0.0,1.141988,0.0
42,Aaron Ramsdale,Manchester City,False,25,5,Arsenal,GK,NaN,0.0,1.439281,0.0
...,...,...,...,...,...,...,...,...,...,...,...
35691,Yerson Mosquera,Southampton,True,25,11,Wolves,DEF,NaN,0.0,1.738830,0.0
35692,Yerson Mosquera,Fulham,False,25,12,Wolves,DEF,NaN,0.0,2.459075,0.0
35693,Yerson Mosquera,Bournemouth,True,25,13,Wolves,DEF,NaN,0.0,2.055266,0.0
35694,Yerson Mosquera,Everton,False,25,14,Wolves,DEF,NaN,0.0,2.135080,0.0


In [13]:
df_plot.groupby('name')[[pred_var, 'value']].mean()

KeyError: "Columns not found: 'value'"

# Team level charts 


In [18]:
df_team = pd.read_csv("/Users/toby/Dev/lionel-app/data/team_selection_1_25.csv")
df_team.head()

,unique_id,team_name,position,value,pred_Naive,pred_LGBMRegressor_no_exog,pred_LSTMWithReLU,picked_Naive,captain_Naive,first_xi_Naive,season,picked_time,picked_LGBMRegressor_no_exog,captain_LGBMRegressor_no_exog,first_xi_LGBMRegressor_no_exog,picked_LSTMWithReLU,captain_LSTMWithReLU,first_xi_LSTMWithReLU
0,Alexander Isak,Newcastle,FWD,84,14.0,4.469617,2.0,1.0,0,1.0,24,2024-08-04 10:09:23.604859,0.0,0,NaN,0.0,0,NaN
1,André Onana,Manchester Utd,GK,50,7.0,3.120548,2.0,1.0,0,1.0,24,2024-08-04 10:09:23.604859,0.0,0,NaN,0.0,0,NaN
2,Bruno Guimarães Rodriguez Moura,Newcastle,MID,58,16.0,5.171441,2.0,1.0,0,1.0,24,2024-08-04 10:09:23.604859,1.0,0,1.0,0.0,0,NaN
3,Dejan Kulusevski,Tottenham,MID,67,16.0,2.760795,2.0,1.0,0,1.0,24,2024-08-04 10:09:23.604859,0.0,0,NaN,0.0,0,NaN
4,Diogo Dalot Teixeira,Manchester Utd,DEF,52,15.0,3.661784,1.0,1.0,0,1.0,24,2024-08-04 10:09:23.604859,0.0,0,NaN,0.0,0,NaN


In [19]:
df_team.columns

Index(['unique_id', 'team_name', 'position', 'value', 'pred_Naive',
       'pred_LGBMRegressor_no_exog', 'pred_LSTMWithReLU', 'picked_Naive',
       'captain_Naive', 'first_xi_Naive', 'season', 'picked_time',
       'picked_LGBMRegressor_no_exog', 'captain_LGBMRegressor_no_exog',
       'first_xi_LGBMRegressor_no_exog', 'picked_LSTMWithReLU',
       'captain_LSTMWithReLU', 'first_xi_LSTMWithReLU'],
      dtype='object')

In [24]:
pred_var = "LGBMRegressor_no_exog"
df_team[df_team[f"picked_{pred_var}"] == 1].head()

,unique_id,team_name,position,value,pred_Naive,pred_LGBMRegressor_no_exog,pred_LSTMWithReLU,picked_Naive,captain_Naive,first_xi_Naive,season,picked_time,picked_LGBMRegressor_no_exog,captain_LGBMRegressor_no_exog,first_xi_LGBMRegressor_no_exog,picked_LSTMWithReLU,captain_LSTMWithReLU,first_xi_LSTMWithReLU
2,Bruno Guimarães Rodriguez Moura,Newcastle,MID,58,16.0,5.171441,2.0,1.0,0,1.0,24,2024-08-04 10:09:23.604859,1.0,0,1.0,0.0,0,NaN
7,Jarell Quansah,Liverpool,DEF,40,15.0,3.777053,0.0,1.0,0,1.0,24,2024-08-04 10:09:23.604859,1.0,0,0.0,0.0,0,NaN
10,Phil Foden,Manchester City,MID,85,15.0,6.284066,3.0,1.0,0,1.0,24,2024-08-04 10:09:23.604859,1.0,0,1.0,0.0,0,NaN
12,Guglielmo Vicario,Tottenham,GK,53,6.0,4.000374,2.0,1.0,0,0.0,24,2024-08-04 10:09:23.604859,1.0,0,1.0,0.0,0,NaN
89,Benjamin White,Arsenal,DEF,61,2.0,5.023394,4.0,0.0,0,NaN,24,2024-08-04 10:09:23.604859,1.0,0,1.0,1.0,0,1.0


In [34]:
fig = go.Figure()
df_not_picked = df_team[df_team[f"picked_{pred_var}"] == 0]
df_picked = df_team[df_team[f"picked_{pred_var}"] == 1]


In [39]:
fig.add_trace(go.Scatter(
    x=df_not_picked[df_not_picked[f"picked_{pred_var}"] == 0].value,
    y=df_not_picked[df_not_picked[f"picked_{pred_var}"] == 0][f"pred_{pred_var}"],
    marker=dict(
        color='#9fbbe3', 
    ),
    mode='markers',
    customdata=df_not_picked[["unique_id", "team_name", f"pred_{pred_var}"]],
            hovertemplate="<b>%{customdata[0]}</b>"
            + "<br><br><b>Team:</b> %{customdata[1]}"
            + "<br><b>Forecasted Points:</b> %{customdata[2]}"
            + "<extra></extra>",
))

In [40]:
fig.add_trace(go.Scatter(
            x=df_picked.value,
            y=df_picked[f"pred_{pred_var}"],
            marker=dict(
                color='#4B5563', 
            ),
            mode='markers',
            customdata=df_picked[["unique_id", "team_name", f"pred_{pred_var}"]],
            hovertemplate="<b>%{customdata[0]}</b>"
            + "<br><br><b>Team:</b> %{customdata[1]}"
            + "<br><b>Forecasted Points:</b> %{customdata[2]}"
            + "<extra></extra>",))

fig.update_layout(
    autosize=False,
    width=700,
    height=800,
    showlegend=False,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    xaxis_title="Player Value (as of previous GW)",
    yaxis_title="Forecasted Points",
    yaxis_visible=True, 
    yaxis_showticklabels=False,
    xaxis_visible=True, 
    xaxis_showticklabels=False,
    font=dict(
        family="sans-serif",
        color="#4B5563"
    )
)

# Check gameweek

In [36]:
import datetime as dt
today = dt.datetime.today().date()

df = pd.DataFrame(dbm.query(f"SELECT * FROM fixtures WHERE season = {25}").fetchall())
df = df.groupby('gameweek').agg(first_kickoff=('kickoff_time', 'min'), last_kickoff=('kickoff_time', 'max')).reset_index()
df[['first_kickoff', 'last_kickoff']] = df[['first_kickoff', 'last_kickoff']].apply(pd.to_datetime)
df[df.last_kickoff.dt.date < today].iloc[-1, 0] + 1


5

In [37]:
df[df.last_kickoff.dt.date < today]

,gameweek,first_kickoff,last_kickoff
0,1,2024-08-16 19:00:00,2024-08-19 19:00:00
1,2,2024-08-24 11:30:00,2024-08-25 15:30:00
2,3,2024-08-31 11:30:00,2024-09-01 15:00:00
3,4,2024-09-14 11:30:00,2024-09-15 15:30:00


In [31]:
df.last_kickoff.dt.date < today

0      True
1      True
2      True
3      True
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
24    False
25    False
26    False
27    False
28    False
29    False
30    False
31    False
32    False
33    False
34    False
35    False
36    False
37    False
Name: last_kickoff, dtype: bool